# 12 · Sombras de la regresión logística

El notebook 11 compara la logística con una referencia ingenua. Ahora examinamos cómo se sitúa su mejora frente a versiones que desplazan las características en el tiempo, manteniendo las etiquetas en sus fechas originales.

**Antes de ejecutar:** deben estar completas las tablas de 01 → 02 → 07 → 08 → 10 → 11. El 09 no es una entrada. Utiliza el mismo entorno `.venv` del 11; no hacen falta dependencias nuevas.

Solo se utiliza **1927–1989**. El [protocolo](../docs/sombras_logistica.md) se fija tras conocer los resultados iniciales del 11 y antes de calcular sus sombras. Este cuaderno no vuelve a entrenar el modelo original ni cambia su configuración.


## 1. Qué es una sombra

Imagina una tabla con dos partes: las condiciones anteriores del mercado y el resultado posterior de cada señal. Deslizamos la parte de condiciones, mientras los resultados permanecen en sus filas. Al llegar a un extremo, las condiciones que salen vuelven por el otro. Así obtenemos una referencia que altera su coincidencia temporal.

Las diez características se mueven juntas, conservando su relación entre sí. El mismo desplazamiento sirve para momentum y reversión. Las etiquetas, señales y fechas de CPCV permanecen fijas. Primero desplazamos sobre todos los meses comunes; después utilizamos los meses con señal. Así, los meses en efectivo de momentum siguen ocupando su lugar en el calendario.

Estas referencias son artificiales: el empalme entre extremos no representa continuidad histórica y algunas condiciones trasladadas proceden de fechas posteriores. No son predicciones ejecutables en sus nuevas fechas. Los cambios de régimen y la relación entre rotaciones limitan su interpretación estadística.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from tfm.data import project_root, read_result, save_result
from tfm.logistic import load_logistic_inputs
from tfm.model_shadows import (MIN_SHIFT, SHADOW_QUANTILE, QUANTILE_METHOD,
                               model_calendar, circular_shifts, run_model_shadows)

ROOT = project_root()
ejemplos, particiones, _, _ = load_logistic_inputs(ROOT)
calendario = model_calendar(ejemplos)
desplazamientos = circular_shifts(len(calendario))
metricas_originales, _ = read_result(ROOT, "logistic_metrics")
display(metricas_originales.loc[metricas_originales.aggregation.eq("monthly_mean"),
                               ["strategy", "model", "examples", "log_loss", "auc"]])
display(pd.Series({"Primer mes común": calendario.target_period.iloc[0],
                   "Último mes común": calendario.target_period.iloc[-1],
                   "Meses consecutivos": len(calendario),
                   "Distancia circular mínima": MIN_SHIFT,
                   "Sombras previstas": len(desplazamientos),
                   "Ajustes previstos (15 particiones × 2 reglas por sombra)": len(desplazamientos) * 30},
                  name="Diseño fijado"))


## 2. Por qué la distancia se mide en ambos sentidos

Un desplazamiento de un mes y otro de todos los meses menos uno dejan las series a un mes de distancia, solo que en direcciones opuestas. Por eso exigimos al menos 36 meses en cualquiera de los sentidos.

Con 713 meses se utilizan todos los desplazamientos enteros desde 36 hasta 677, ambos incluidos: **642 sombras**. No se escogen después las más favorables. La tabla siguiente muestra el origen de las características en una rotación; las fechas objetivo y las etiquetas siguen en su sitio. La transformación causal original no se vuelve a calcular sobre la serie desplazada.


In [ ]:
desplazamiento_ejemplo = desplazamientos[0]
ejemplo_origen = pd.DataFrame({
    "Mes objetivo conservado": calendario.target_period,
    "Mes del que proceden las características de la sombra": np.roll(
        calendario.target_period.to_numpy(), desplazamiento_ejemplo)})
display(ejemplo_origen.head(6))
print("Primer desplazamiento:", desplazamientos[0], "| Último:", desplazamientos[-1])


## 3. Repetir el mismo aprendizaje

Cada sombra usa la misma logística L2 con C=1 y las mismas quince particiones, purga y embargo. Se ajusta solo con el entrenamiento de cada partición. Para cada ejemplo se conservan cinco predicciones de validación y se evalúa su media, igual que en el 11. La referencia ingenua sigue dependiendo de las etiquetas originales de entrenamiento, por lo que debe coincidir con la referencia del modelo original.

La siguiente celda ejecuta el recorrido completo y puede tardar varios minutos. Muestra su progreso cada 25 sombras. Registra el inicio y la terminación o fallo en `outputs/experiments/logistic_shadow_trials.jsonl`, separado de las configuraciones candidatas del 11. Una excepción o una interrupción detiene y registra el intento, sin cambiar parámetros automáticamente.

No se guardan los millones de predicciones intermedias: se conserva el resumen de cada sombra. Las particiones sin ejemplos y los ajustes no disponibles conservan sus estados. Si falta alguna sombra válida, la comparación completa queda no disponible; no se calcula el percentil excluyendo los fallos.


In [ ]:
def mostrar_progreso(completadas, total):
    print(f"Sombras evaluadas: {completadas}/{total}", flush=True)

tablas, informe = run_model_shadows(ROOT, progress=mostrar_progreso)
sombras = tablas["logistic_shadows"]
comparacion = tablas["logistic_shadow_comparison"]
print("Estado del recorrido:", informe["experiment_status"])
print("Comparación completa disponible:", informe["shadow_comparison_available"])
display(sombras.loc[sombras.shift_months.eq(desplazamientos[0])].rename(columns={
    "strategy": "Regla", "shift_months": "Desplazamiento", "log_loss": "Log-loss",
    "auc": "AUC", "log_loss_improvement": "Mejora frente al ingenuo", "status": "Estado"}))


## 4. Cómo se realiza la comparación

La medida principal es **log-loss del ingenuo menos log-loss del modelo**. Una mejora positiva indica probabilidades mejores que las de esa referencia; cuanto mayor, mejor.

Calculamos el percentil 95 de las mejoras de las sombras utilizando el cuantil por orden superior (`higher`). La mejora original debe superar estrictamente ese umbral. La tabla muestra también la AUC y su comparación descriptiva con las sombras.

El criterio operativo combina AUC original de al menos 0,54, mejora positiva frente al ingenuo y mejora superior al percentil 95 de sombras. La AUC de las sombras es información complementaria, sin añadir otro requisito automático. La fracción de sombras que queda por debajo del resultado original no es un valor p exacto ni una probabilidad de que la mejora sea real.


In [ ]:
columnas = {
    "strategy": "Regla", "examples": "Ejemplos", "original_auc": "AUC original",
    "original_improvement": "Mejora original de log-loss",
    "improvement_threshold_95": "Percentil 95 de mejora de sombras",
    "completed_shadows": "Sombras completas", "auc_at_least_054": "AUC al menos 0,54",
    "beats_naive_log_loss": "Mejora frente al ingenuo",
    "exceeds_shadow_threshold": "Supera sombras",
    "meets_operational_criterion": "Cumple criterio operativo", "status": "Estado"}
display(comparacion[list(columnas)].rename(columns=columnas))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, regla in zip(axes, ("momentum", "reversal")):
    fila = comparacion.loc[comparacion.strategy.eq(regla)].iloc[0]
    ax.set(title=regla, xlabel="Mejora de log-loss frente al ingenuo", ylabel="Número de desplazamientos")
    if fila.comparison_available:
        valores = sombras.loc[sombras.strategy.eq(regla), "log_loss_improvement"]
        ax.hist(valores, bins=35, color="#a9cbe4", edgecolor="white", label="Sombras")
        ax.axvline(fila.improvement_threshold_95, color="#b24b37", linestyle="--", label="Percentil 95")
        ax.axvline(fila.original_improvement, color="#163e64", linewidth=2, label="Modelo original")
        ax.legend()
    else:
        ax.text(.5, .5, "Comparación incompleta: sin umbral", ha="center", transform=ax.transAxes)
    ax.grid(alpha=.2)
fig.tight_layout()
plt.show()


## 5. AUC como información complementaria

La tabla muestra dónde queda la AUC original frente a las AUC de las sombras. Su percentil 95 no forma parte de un segundo filtro automático: hemos fijado la comparación principal mediante log-loss. Las dos medidas describen aspectos diferentes de las probabilidades y pueden ofrecer resultados distintos.

Las sombras comparten datos y están relacionadas entre sí. Tampoco los cinco modelos que predicen cada mes son observaciones independientes. No interpretamos 642 rotaciones como 642 pruebas independientes ni afirmamos controlar conjuntamente un error del 5 % para ambas reglas.


In [ ]:
display(comparacion[["strategy", "original_auc", "auc_threshold_95_descriptive",
                     "fraction_shadows_below_original_auc",
                     "fraction_shadows_below_original_improvement", "comparison_available"]]
        .rename(columns={"strategy": "Regla", "original_auc": "AUC original",
            "auc_threshold_95_descriptive": "Percentil 95 de AUC (descriptivo)",
            "fraction_shadows_below_original_auc": "Fracción de sombras con AUC menor",
            "fraction_shadows_below_original_improvement": "Fracción con mejora de log-loss menor",
            "comparison_available": "Comparación disponible"}))


## 6. Qué permite concluir y qué queda pendiente

Cumplir este criterio indica que el resultado destaca frente a las referencias circulares elegidas. No demuestra causalidad, estabilidad futura o rentabilidad después de costes de una cartera filtrada. Si no se cumple, se documentará sin retocar los desplazamientos o los parámetros para lograrlo.

Continúan pendientes Clustered MDA con validación interna, la comparación con otros candidatos y SHAP. Antes de evaluar el filtro como inversión se fijarán sus decisiones de aceptación y se simulará una cuenta continua. No se calcula aquí DSR/PBO ni se utiliza el periodo desde 1990.

La celda final guarda dos tablas Parquet y sus JSON en `data/processed/`. Si falla, repítela completa antes de utilizar las salidas; no necesitas repetir los ajustes para guardar de nuevo. Las tablas originales del 11 y su historial se conservan. Repetir el 11 retira las salidas del 12, que deberán recalcularse. Antes del commit, limpia las salidas del notebook y guarda.

Referencia: guía del tutor del 07/11/2026, anexo de sombras y sublínea 1.B. Los detalles operativos de la comparación se justifican en [el protocolo](../docs/sombras_logistica.md); el entrenamiento y las métricas reutilizan el notebook 11.


In [ ]:
guardados = []
for nombre, tabla in tablas.items():
    salida = save_result(ROOT, nombre, tabla, informe)
    guardados.append({"Tabla": nombre + ".parquet", "Filas": salida["rows"]})
display(pd.DataFrame(guardados))
print("Historial de sombras:", informe["trial_log"])
print("Las salidas y el historial del modelo original se conservan.")
